# Week 15, Day 5: Simulating randomness: inverse transforms, thinning and dropout

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w015/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: From uniform numbers to anything

Write `sample_exponential(u, r)` and `sample_discrete(pmf, u)` (use the cumulative sum and np.searchsorted). Then sample 20,000 tokens from the softmax of the logits at temperatures 0.5, 1 and 2, and store the frequency of the top token at each temperature in the dictionary `top_freq`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(1979)

def sample_exponential(u, r):
    return ...

def sample_discrete(pmf, u):
    return ...                     # the index of the first cumulative probability above u

def softmax(z, tau):
    e = np.exp((z - z.max()) / tau)
    return e / e.sum()

logits = np.array([2.0, 1.5, 0.5, 0.0, -1.0])
top_freq = {}
for tau in [0.5, 1.0, 2.0]:
    tokens = sample_discrete(softmax(logits, tau), rng.random(20000))
    top_freq[tau] = ...
print("frequency of the top token at temperatures 0.5, 1, 2:", {t: round(f, 3) for t, f in top_freq.items()})

In [ ]:
import numpy as np
def test_exponential():
    "sample_exponential gives mean 1/r and coefficient of variation 1"
    x = sample_exponential(np.random.default_rng(0).random(200000), 20)
    assert abs(x.mean() - 0.05) < 0.001 and abs(x.std() / x.mean() - 1) < 0.02

def test_discrete():
    "sample_discrete reproduces the PMF"
    pmf = np.array([0.1, 0.6, 0.3])
    k = sample_discrete(pmf, np.random.default_rng(1).random(100000))
    assert np.allclose(np.bincount(k, minlength=3) / len(k), pmf, atol=0.01)

def test_temperature():
    "lower temperature makes the top token more frequent"
    assert top_freq[0.5] > top_freq[1.0] > top_freq[2.0]

test_exponential()
test_discrete()
test_temperature()
print("All checks passed")

### Exercise: Spikes from a time-varying rate

Write `thinning(rate, lmax, T, rng)` that returns an array of spike times. Generate 300 trials of a rate rising and falling as a stimulus comes on, build a PSTH in 50 ms bins (spikes per second), and compare the mean count with the integral of the rate.

In [ ]:
def thinning(rate, lmax, T, rng):
    times = []
    t = 0.0
    while True:
        t += ...                                 # exponential interval at rate lmax
        if t > T:
            break
        if ...:                                  # keep with probability rate(t) / lmax
            times.append(t)
    return np.array(times)

T = 2.0
def rate(t): return 5 + 45 * np.exp(-((t - 0.8) / 0.15) ** 2)      # a transient response to a stimulus at 0.8 s
lmax = 50
trials = [thinning(rate, lmax, T, rng) for _ in range(300)]
edges = np.arange(0, T + 1e-9, 0.05); centres = (edges[:-1] + edges[1:]) / 2
psth = np.histogram(np.concatenate(trials), edges)[0] / (300 * 0.05)
tt = np.linspace(0, T, 4001)
expected_count = np.sum(rate(tt[:-1]) * np.diff(tt))
print(f"mean count {np.mean([len(s) for s in trials]):.2f}, integral of the rate {expected_count:.2f}")
plt.bar(centres, psth, width=0.045, alpha=0.5, label="PSTH"); plt.plot(tt, rate(tt), "k", label="rate"); plt.legend(); plt.xlabel("time (s)"); plt.show()

In [ ]:
import numpy as np
def test_counts():
    "the mean count per trial matches the integral of the rate (to within 3%)"
    assert abs(np.mean([len(s) for s in trials]) - expected_count) / expected_count < 0.03

def test_psth():
    "the PSTH follows the rate: correlation above 0.95"
    assert np.corrcoef(psth, rate(centres))[0, 1] > 0.95

def test_in_range():
    "all spike times lie in [0, T]"
    assert all((s.min() >= 0) and (s.max() <= T) for s in trials if len(s))

test_counts()
test_psth()
test_in_range()
print("All checks passed")

### Exercise: At least three of eight

Release probabilities differ across 8 sites. Estimate P(at least 3 release) by simulating 20,000 trials (`p_mc`) with its standard error `se_mc`. Compute the exact value `p_exact` by convolving the eight Bernoulli PMFs [1 − p, p].

In [ ]:
p_sites = np.array([0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.6, 0.8])
n_sims = 20000
released = rng.random((n_sims, 8)) < p_sites            # True where a site released
p_mc = ...
se_mc = ...

pmf = np.array([1.0])
for p in p_sites:
    pmf = np.convolve(pmf, [1 - p, p])                   # PMF of the number released so far
p_exact = ...
print(f"Monte Carlo {p_mc:.4f} ± {1.96 * se_mc:.4f} (95%),  exact {p_exact:.4f}")

In [ ]:
import numpy as np
def test_exact():
    "the exact probability, by convolving the Bernoulli PMFs, is a valid probability"
    assert 0 < p_exact < 1

def test_covers():
    "the Monte Carlo 95% interval contains the exact value"
    assert abs(p_mc - p_exact) < 1.96 * se_mc * 1.5

def test_se():
    "the standard error is computed from p_mc and n"
    assert abs(se_mc - np.sqrt(p_mc * (1 - p_mc) / n_sims)) < 1e-12

test_exact()
test_covers()
test_se()
print("All checks passed")

### Exercise: Check the dropout moments

For a unit with 100 inputs, simulate 50,000 inverted-dropout masks with keep probability 0.8, compute the input `u` on each, and compare its mean and variance with the formulas (`var_pred`).

In [ ]:
p_keep = 0.8
x = rng.exponential(1.0, 100)              # one input pattern
w = rng.normal(0, 0.1, 100)
masks = rng.random((50000, 100)) < p_keep
u = ...
var_pred = ...
print(f"mean {u.mean():.4f} (w.x = {w @ x:.4f}),  variance {u.var():.5f} (predicted {var_pred:.5f})")

In [ ]:
import numpy as np
def test_mean():
    "the simulated mean input matches w . x to within 1%"
    assert abs(u.mean() - w @ x) / abs(w @ x) < 0.01

def test_var():
    "the simulated variance matches (1 - p) / p * sum (w x)^2 to within 3%"
    assert abs(u.var() - var_pred) / var_pred < 0.03

test_mean()
test_var()
print("All checks passed")